# ImmunoForge parameter reference

This notebook documents the active command-line arguments and every active configuration section in `config/default_config.yaml`.

- Scope: CLI usage + pipeline config
- Source of truth: `immunoforge/cli.py` and `config/default_config.yaml`
- Current note: legacy Web UI parameters were removed from the package and are not part of the active config surface anymore.


## 1. Top-level CLI help

Use any of the following forms:

- `immunoforge --help`
- `immunoforge -h`
- `immunoforge --h`

The same help flags also work on subcommands such as `immunoforge run --h`.


## 2. CLI argument reference

### `immunoforge run`

| Argument | Purpose | Typical usage |
| --- | --- | --- |
| `-c`, `--config PATH` | Selects the YAML configuration file. | Point to a campaign-specific config when you do not want to edit `config/default_config.yaml`. |
| `-s`, `--steps ...` | Restricts execution to specific pipeline steps. | Use `B1`, `B3b`, `B5`, `B6`, etc. for smoke tests or partial reruns. |
| `--species NAME` | Overrides `species.default` at runtime. | Fast one-off swap between `mouse`, `human`, and `cynomolgus`. |
| `-o`, `--output DIR` | Overrides `paths.output_dir` at runtime. | Keep experimental runs isolated in a separate output folder. |

### `immunoforge targets`

| Argument | Purpose | Typical usage |
| --- | --- | --- |
| `--cell-type TEXT` | Filters targets by biological context. | Example: `T cell`, `NK cell`, `DC`, `macrophage`, `tumour`. |
| `--species TEXT` | Filters the database by species. | Narrow output to the species you can experimentally validate. |
| `--benchmark` | Shows benchmark-tagged targets only. | Quick browse of curated reference targets. |

### `immunoforge qc`

| Argument | Purpose | Typical usage |
| --- | --- | --- |
| `FASTA` | Input sequence file. | Use one or more protein records in FASTA format. |
| `-o`, `--output PATH` | Optional JSON export. | Save machine-readable QC results for downstream filtering. |

### `immunoforge codon`

| Argument | Purpose | Typical usage |
| --- | --- | --- |
| `sequence` | Amino-acid sequence or `@file.fasta`. | Use `@file.fasta` to read the first record from a FASTA file. |
| `--species` | Selects the codon usage table. | Choose `mouse`, `human`, or `cynomolgus`. |
| `--system` | Selects the cassette builder branch. | Choose `vaccinia`, `aav`, `lentivirus`, `mrna`, `cho`, or `hek293t`. |


## 3. Configuration usage pattern

Edit `config/default_config.yaml` when you want persistent defaults for a campaign. Use CLI overrides when you want temporary per-run changes.

Recommended workflow:

1. Copy `config/default_config.yaml` to a campaign-specific file.
2. Edit target definitions and scoring thresholds in the YAML file.
3. Use `immunoforge run --config your_config.yaml` for reproducible reruns.
4. Use `--species` and `--output` only for temporary overrides.


## 4. Core metadata and target definition

### `project`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `name` | Project label embedded in outputs and metadata. | Keep stable across reruns of the same campaign. |
| `version` | Human-readable config version. | Increment when major thresholds or target sets change. |
| `description` | Free-text description of the run. | Useful for audit trails and exports. |

### `species`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `default` | Default organism used by the pipeline. | Should match the primary codon / biological context for the run. |
| `available` | Allowed species list for the config. | Keep aligned with what the codebase actually supports. |

### `targets.<target_name>`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `name` | Display name of the target. | Use a stable, human-readable identifier. |
| `uniprot` | UniProt accession. | Helps keep structure downloads and reporting traceable. |
| `source` | Structure source: `alphafold` or `pdb`. | Use `pdb` when an experimental structure is available. |
| `pdb_id` | PDB accession when `source=pdb`. | Leave `null` for AlphaFold-based runs. |
| `af_id` | AlphaFold structure ID. | Use when `source=alphafold`. |
| `chain` | Chain to analyze. | Must match the chosen structure file. |
| `residue_range` | Domain or ectodomain boundaries. | Restricts analysis to the biologically relevant segment. |
| `hotspot_residues` | Manual interface guidance residues. | Use when prior biology suggests a known epitope or hotspot zone. |
| `organism` | Organism label for reporting. | Keep consistent with `species.default`. |
| `cell_type` | Biological annotation of the target. | Used for browsing and reporting rather than scoring. |


## 5. Generation and QC controls

### `rfdiffusion`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `model` | RFdiffusion model name. | Keep synced with the installed checkpoint set. |
| `model_dir` | Filesystem path to model weights. | Must exist in the execution environment. |
| `n_designs_per_target` | Number of backbones to generate per target. | Increase for harder targets; decreases speed. |
| `binder_length` | Allowed binder length range. | Broader ranges improve diversity but expand search space. |
| `diffusion_steps` | Number of denoising steps. | Higher can improve structure quality at extra cost. |
| `noise_scale_ca` | CA-coordinate noise scale. | Tune cautiously; larger values broaden sampling. |
| `noise_scale_frame` | Frame noise scale. | Same tradeoff as `noise_scale_ca`. |
| `hotspot_guidance` | Whether manual / predicted hotspots guide design. | Usually keep enabled for receptor-focused campaigns. |
| `dual_target_designs` | Number of dual-target attempts. | Use only for bridging or dual-specificity workflows. |
| `dual_strategies` | Number of dual-target strategy variants. | Increase only if dual-target design is a priority. |

### `proteinmpnn`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `model` | ProteinMPNN checkpoint family. | `soluble` is a reasonable default for secretion-friendly proteins. |
| `sampling_temperature` | Sequence diversity control. | Lower values are more conservative; higher values increase diversity. |
| `sequences_per_backbone` | Number of sampled sequences per backbone. | Increase when you want broader sequence exploration. |
| `design_chain_only` | Restricts redesign to the binder chain. | Usually keep `true` to preserve target structure. |

### `sequence_qc`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `protease_checks.furin.pattern` | Rejects furin-like cleavage motifs. | Important for secreted constructs. |
| `protease_checks.thrombin.pattern` | Rejects thrombin motif. | Helps avoid unintended cleavage liabilities. |
| `protease_checks.enterokinase.pattern` | Rejects enterokinase motif. | Same principle as other hard cleavage filters. |
| `protease_checks.mmp_motifs.patterns` | Tracks MMP-prone motif families. | Use density-based filtering instead of hard motif rejection. |
| `protease_checks.mmp_motifs.max_density_pct` | Maximum allowed MMP motif density. | Lower values are stricter. |
| `protease_checks.cathepsin_b_gflg.pattern` | Rejects the classic GFLG motif. | Useful for stability-sensitive biologics. |
| `protease_checks.cathepsin_b_dibasic.patterns` | Tracks dibasic cleavage motifs. | Common liability screen in secreted proteins. |
| `protease_checks.cathepsin_b_dibasic.max_density_pct` | Allowed dibasic motif density. | Reduce if cleavage susceptibility is a concern. |
| `toxicity.poly_cationic.pattern` | Rejects long basic stretches. | Helps avoid membrane / toxicity liabilities. |
| `toxicity.nls_signal.pattern` | Rejects strong NLS-like patterns. | Useful when nuclear localization is undesirable. |
| `aggregation.apr_pattern` | Aggregation-prone region pattern. | Conservative hydrophobic stretch filter. |
| `aggregation.action` | QC action when APR is detected. | Typically `reject`. |
| `cysteine.require_even` | Enforces even cysteine count. | Reduces orphan-disulfide risk. |
| `isoelectric_point.min_pi` | Lower pI bound. | Raise if acidic candidates create expression issues. |
| `isoelectric_point.max_pi` | Upper pI bound. | Lower if very basic candidates create developability issues. |


## 6. Validation and optimization controls

### `structure_validation`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `method` | Validation backend selection. | `auto` lets the pipeline choose the available path. |
| `dual_evaluation` | Combines structural confidence with sequence plausibility checks. | Usually keep enabled. |
| `high_threshold.mean_plddt` | Mean pLDDT cutoff for HIGH confidence. | Raise for stricter structure quality. |
| `high_threshold.fraction_above_70` | Fraction of residues above pLDDT 70 for HIGH confidence. | Adds robustness beyond the mean. |
| `medium_threshold.mean_plddt` | Mean pLDDT cutoff for MEDIUM confidence. | Relaxed fallback threshold. |
| `medium_threshold.fraction_above_70` | Residue-fraction threshold for MEDIUM confidence. | Lower values allow marginal candidates through. |
| `pass_levels` | Accepted confidence classes. | Restrict to `HIGH` for maximum stringency. |

### `hotspot_prediction`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `enabled` | Enables automatic hotspot prediction. | Useful when manual hotspot residues are unavailable. |
| `method` | Prediction backend. | `auto` is safest unless a specific backend is validated. |
| `top_k` | Number of residues to keep as hotspots. | Too high can dilute guidance; too low can miss interfaces. |

### `sequence_optimization`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `enabled` | Enables B3a multi-stage optimization. | Turn off for faster, simpler runs. |
| `temperatures` | ProteinMPNN sampling schedule. | Higher-to-lower progression balances diversity and refinement. |
| `seqs_per_temperature` | Number of sequences sampled at each temperature. | Increase for broader exploration. |
| `helicity_weight` | Bias toward helical solutions. | Most useful for miniprotein / helical designs. |
| `stage2_top_k` | Number of candidates kept after stage 2 scoring. | Higher keeps diversity at extra compute cost. |
| `stage3_mutations_per_candidate` | Number of greedy mutations explored per candidate. | Increase only if interface refinement is a bottleneck. |
| `stage3_interface_bias` | Probability of mutating an interface position. | Higher values focus refinement on binding contacts. |
| `stage4_top_n` | Final number of candidates kept per backbone. | Keep small for manageable downstream validation. |


## 7. Scoring, ranking, and DNA design

### `affinity`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `methods` | Active affinity estimators. | Keep aligned with the methods installed in your environment. |
| `consensus` | Consensus aggregation rule. | `geometric_mean` is the current default. |
| `hotspot_analysis` | Enables residue-level hotspot interpretation. | Helpful for diagnosis and reporting. |
| `confidence_threshold_log10` | Confidence band threshold in log10 nM units. | Lower is stricter. |

### `ranking`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `weights.plddt` | Structural confidence contribution. | Increase if foldability is the main priority. |
| `weights.ddg` | Energy / stability contribution. | Useful when interface energetics dominate. |
| `weights.kd` | Affinity contribution. | Increase when potency is the main screen. |
| `weights.mpnn_score` | Sequence-model plausibility contribution. | Helps suppress unlikely sequences. |
| `weights.bsa` | Buried surface area contribution. | Interface size proxy. |
| `weights.shape_complementarity` | Interface fit contribution. | Useful for docking-style refinement. |
| `weights.baseline` | Baseline score offset. | Small stabilizer term for composite scoring. |
| `penalties.high_aggregation` | Penalty for aggregation-prone candidates. | Use more negative values for stricter filtering. |
| `penalties.extreme_pi` | Penalty for out-of-range pI. | Mild developability correction. |
| `top_n` | Number of ranked candidates retained. | Controls how many proceed to later steps. |
| `synthesis_top_n` | Number of top candidates recommended for synthesis. | Keep small for experimental focus. |

### `codon_optimization`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `species_tables.*` | Mapping from species name to codon-usage table. | Extend only with validated codon tables. |
| `gc_content_target` | Desired CDS GC fraction range. | Narrow ranges improve manufacturability but limit flexibility. |
| `avoid_patterns.vaccinia_t5nt` | Pattern removed for vaccinia compatibility. | Specific to the vaccinia cassette branch. |
| `restriction_sites_avoid` | Restriction sites removed from output DNA. | Match cloning strategy and synthesis constraints. |
| `signal_peptides.il2_leader` | Named leader peptide for secretion. | Useful when IL-2 leader is experimentally preferred. |
| `signal_peptides.igk_leader` | Alternative secretion leader. | Common antibody-compatible choice. |
| `expression_systems.vaccinia.promoter` | Vaccinia promoter label. | Metadata and cassette rendering support. |
| `expression_systems.vaccinia.kozak` | Vaccinia translation-initiation sequence. | Usually keep unchanged. |
| `expression_systems.vaccinia.insertion_locus` | Vaccinia integration locus. | Informational unless downstream tooling consumes it. |
| `expression_systems.vaccinia.cassette_format` | Vaccinia cassette layout description. | Human-readable cassette schema. |
| `expression_systems.aav.promoter` | AAV promoter label. | Keep aligned with vector design. |
| `expression_systems.aav.kozak` | AAV kozak sequence. | Typically unchanged. |
| `expression_systems.lentivirus.promoter` | Lentiviral promoter label. | Match the intended transfer vector. |
| `expression_systems.lentivirus.kozak` | Lentiviral kozak sequence. | Typically unchanged. |


## 8. Output paths and advanced modules

### `paths`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `output_dir` | Root folder for run outputs. | The main runtime override target for `--output`. |
| `data_dir` | General data root. | Keep stable across runs if you cache shared assets. |
| `structures_dir` | Structure-cache location. | Useful for avoiding repeated downloads. |
| `logs_dir` | Log-file directory. | Usually keep nested under `output_dir`. |

### `af3_validation`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `enabled` | Enables B9a AF3 / Boltz-2 validation. | Turn off when no compatible environment is available. |
| `top_n` | Number of ranked candidates to validate. | Limit to preserve GPU budget. |
| `recycling_steps` | Boltz recycling iterations. | More recycling can improve structure confidence at extra runtime cost. |
| `sampling_steps` | Diffusion sampling steps. | Higher values increase compute. |
| `diffusion_samples` | Number of structure samples per candidate. | Increase only if structural diversity matters. |
| `timeout_per_prediction` | Hard timeout in seconds. | Protects long jobs from hanging indefinitely. |

### `af2_filter`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `enabled` | Enables B4c AF2-multimer filtering. | Turn off if AF2 is unavailable. |
| `max_candidates` | Max candidates scored with AF2. | Caps expensive orthogonal validation. |
| `iptm_reject_threshold` | Hard reject threshold for ipTM. | Raise for stricter structure-quality filtering. |
| `iptm_high_threshold` | HIGH-confidence ipTM threshold. | Used to classify stronger structures. |

### `affinity_maturation`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `enabled` | Enables B9b conditional maturation. | Turn off for simpler first-pass screens. |
| `kd_threshold_nM` | Candidates weaker than this enter maturation. | Higher values trigger maturation more often. |
| `target_kd_nM` | Desired post-maturation affinity. | Sets the optimization target. |
| `max_generations` | Maximum maturation rounds. | Increase only if search budget allows. |
| `candidates_per_gen` | Candidates explored per round. | Main breadth control. |
| `top_k_per_gen` | Survivors carried forward per round. | Lower values are more selective. |
| `temperatures` | Resampling temperatures for maturation. | Include a conservative and a more exploratory value. |
| `seed` | Random seed for reproducibility. | Keep fixed for auditability. |

### `denovo_profile`

| Parameter | Purpose | Guidance |
| --- | --- | --- |
| `binder_type` | Activates the de novo calibration branch. | Must stay `de_novo` for miniprotein calibration. |
| `rfdiffusion.*` | Overrides generation behavior for de novo campaigns. | Shorter binders and more designs improve hit rate. |
| `proteinmpnn.*` | Overrides sequence sampling for de novo campaigns. | Often uses more sequences and slightly higher diversity. |
| `sequence_optimization.*` | Overrides B3a optimization intensity. | Higher search depth for compact binders. |
| `affinity_maturation.*` | Overrides maturation aggressiveness. | Targets tighter binders more aggressively. |
| `ranking.weights.*` | Rebalances scoring for de novo candidates. | Often gives more weight to affinity and fold confidence. |


## 9. Practical editing examples

### Human campaign with a custom output directory

```bash
immunoforge run --config config/default_config.yaml --species human --output outputs_human
```

### QC-only step rerun

```bash
immunoforge run --config config/default_config.yaml --steps B3b
```

### Codon optimization for CHO expression

```bash
immunoforge codon @toy.fasta --species human --system cho
```
